# WS1 Subindex 3 — Critical Mineral Endowment: API Call Pipeline

**Project:** UCSD Rady MSBA Capstone — Chessboard Sovereign Index (CSI)
**Workstream 1 / Subindex 3:** Critical Mineral Endowment

---

## What this notebook does

Fetches all raw data needed to compute SI3 scores for **6 countries × 6 minerals × 5 metrics**:

| Source | Metrics | Granularity | Coverage |
|---|---|---|---|
| **USGS MCS** (ScienceBase API + PDF fallback) | Production Share, Reserves Share, YoY Production Growth | Annual | latest published edition (auto-detected) |
| **UN Comtrade** (`comtradeapicall`) | Refining Capacity (processed exports), Value-Add Ratio | Monthly | 2020-01 to (today − 2 months) |

**Countries (6):** USA, UAE, Brazil, India, Singapore, Philippines
**Minerals (6):** Copper, Lithium, Nickel, Cobalt, Rare Earths, Silicon

## Self-updating design — no hard-coded IDs or end dates

The notebook is built to be **re-runnable indefinitely** as USGS releases new MCS editions
and Comtrade publishes new monthly data. Specifically:

- **MCS edition year**: auto-detected from today's date (USGS publishes annually in late Jan/early Feb).
- **ScienceBase parent item ID**: discovered dynamically via the ScienceBase search API.
  No item ID is hard-coded — when MCS 2027 ships, the same code will find it without changes.
- **Auto-fallback**: if the latest year's release isn't on ScienceBase yet (e.g. running in
  early January), falls back to the prior year up to 3 years back.
- **Comtrade end period**: computed as `today − LAG_MONTHS` (default 2), so each run
  pulls everything available without manual updates.

The only date-related constants are the **start year** (2020, fixed by CSI methodology)
and the lag used for Comtrade availability (configurable; 2 months works as of 2026).

## Output files
- `data/usgs/{mineral}_world_raw.csv` — 6 files
- `data/comtrade/{mineral}_{iso3}_{stage}.csv` — 36 files (processed) + 36 files (raw)
- `data/si3_combined_raw.parquet` — long-format master table
- `data/si3_metric_panel.csv` — wide-format country × mineral × metric panel

## API access
- **USGS**: no key needed — public ScienceBase JSON API + PDF fallback.
- **UN Comtrade**: free tier works out of the box (`previewFinalData()`, 500 records/call).
  Set `COMTRADE_KEY` env var to use the premium endpoint (250K records/call, faster).

---
## 0. Setup & configuration

In [ ]:
# Install dependencies (run once)
# !pip install -q comtradeapicall pandas requests beautifulsoup4 lxml pyarrow openpyxl tqdm

In [ ]:
import os
import re
import time
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import Optional

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Paths ----------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
USGS_DIR     = DATA_DIR / "usgs"
COMTRADE_DIR = DATA_DIR / "comtrade"
LOG_DIR      = DATA_DIR / "logs"

for d in (DATA_DIR, USGS_DIR, COMTRADE_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Comtrade key (optional) ---------------------------------------------
# Register free at https://comtradeplus.un.org for a premium key.
# If COMTRADE_KEY is not set, code falls back to the no-key preview API
# (capped at 500 records per call — fine for 6 countries × monthly × 1 HS code).
COMTRADE_KEY = os.environ.get("COMTRADE_KEY", "").strip() or None
print(f"Comtrade mode: {'PREMIUM (250K rec/call)' if COMTRADE_KEY else 'PREVIEW (500 rec/call, no key)'}")

# --- Run timestamp -------------------------------------------------------
RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
print(f"Run timestamp: {RUN_TS}")

# --- MCS edition auto-detection ------------------------------------------
# USGS publishes MCS{YEAR} in late Jan / early Feb each year, covering
# {YEAR-1} as the latest data year. So:
#   - if today is Feb-Dec of year N, latest available is MCS N
#   - if today is Jan of year N, latest available is still MCS N-1
#     (until USGS publishes the new edition)
# The notebook tries the auto-detected year first; if ScienceBase has no
# matching release yet, it auto-falls back to the prior year.
TODAY = datetime.utcnow()
MCS_YEAR_DEFAULT = TODAY.year if TODAY.month >= 2 else TODAY.year - 1
print(f"Auto-detected MCS edition to try first: {MCS_YEAR_DEFAULT}")
# Override here if you want to force a specific edition:
#   MCS_YEAR_DEFAULT = 2025

---
## 1. Reference tables: countries, minerals, codes

Centralizes all the lookup data so HS codes, M49 codes, and name spellings only live in one place.

In [ ]:
# --- 6 target countries ---------------------------------------------------
# UN M49 numeric codes are required by Comtrade as `reporterCode`.
# ISO3 is included for joining with other CSI workstream data.
COUNTRIES = pd.DataFrame([
    # name,        m49,   iso3,  usgs_aliases (regex; USGS spelling varies)
    ("USA",         "842", "USA", r"United States|United States of America"),
    ("UAE",         "784", "ARE", r"United Arab Emirates|UAE"),
    ("Brazil",      "076", "BRA", r"Brazil"),
    ("India",       "356", "IND", r"India"),
    ("Singapore",   "702", "SGP", r"Singapore"),
    ("Philippines", "608", "PHL", r"Philippines"),
], columns=["country", "m49", "iso3", "usgs_pattern"])
COUNTRIES

In [ ]:
# --- 6 target minerals ---------------------------------------------------
# usgs_slug = the URL slug used by USGS (lowercase, dash-separated).
# mcs_year  = which MCS edition; defaults to MCS_YEAR_DEFAULT (auto-detected).
#             You can override per-mineral if needed (e.g. one mineral
#             didn't publish a CSV in the latest edition).
MINERALS = pd.DataFrame([
    # name,        usgs_slug
    ("Copper",      "copper"),
    ("Lithium",     "lithium"),
    ("Nickel",      "nickel"),
    ("Cobalt",      "cobalt"),
    ("Rare Earths", "rare-earths"),
    ("Silicon",     "silicon"),
], columns=["mineral", "usgs_slug"])
MINERALS["mcs_year"] = MCS_YEAR_DEFAULT
MINERALS

In [ ]:
# --- HS code map (per mineral) -------------------------------------------
# Confirmed in prior project iterations. PROCESSED = refined/refined-stage
# exports (proxy for refining capacity). RAW = ore/concentrate stage
# (used as denominator for value-add ratio).
HS_CODES = {
    "Copper":      {"processed": ["740311","740319","740321","740322","740329"],
                    "raw":       ["260300"]},
    "Lithium":     {"processed": ["282520","283691"],
                    "raw":       ["253090"]},  # spodumene + other lithium minerals
    "Nickel":      {"processed": ["750110","750120","750210","750220"],
                    "raw":       ["260400"]},
    "Cobalt":      {"processed": ["810520","810530"],
                    "raw":       ["260500"]},
    "Rare Earths": {"processed": ["284610","284690"],
                    "raw":       ["253090"]},  # monazite (REE-bearing) — also used by Li
    "Silicon":     {"processed": ["280461","280469"],
                    "raw":       ["262100"]},  # slag/ash containing Si — proxy
}
# Sanity check: every mineral has both keys
assert all(set(v.keys()) == {"processed","raw"} for v in HS_CODES.values())
print(f"HS codes loaded for {len(HS_CODES)} minerals.")

---
## 2. USGS Mineral Commodity Summaries — web scrape

**Strategy:** Each mineral's MCS chapter is a 2-page PDF with two key tables:
1. **World Mine Production and Reserves** (always present; same format across minerals)
2. **Salient Statistics** (US-only; not used for SI3)

The USGS publishes these as both PDFs (`pubs.usgs.gov/periodicals/mcs{year}/mcs{year}-{slug}.pdf`)
and as machine-readable CSVs in an annual ScienceBase data release.

**Self-updating design — no hard-coded IDs or years:**
- Each year's data release has a different ScienceBase parent item ID, so we **discover it dynamically** via the ScienceBase search API (`q=Mineral Commodity Summaries {YEAR} Data Release&format=json`).
- If the auto-detected year's release isn't published yet (e.g., running in early January), we automatically fall back to the prior year.
- PDF URLs follow a stable pattern (`mcs{year}/mcs{year}-{slug}.pdf`) and only change with the year.

This notebook tries the **ScienceBase CSV** route first (cleaner, structured), and falls back to
**PDF text extraction** if the CSV lookup fails.

In [ ]:
# --- Stable URL templates (only the {year}/{slug} placeholders change) ---
SCIENCEBASE_API = "https://www.sciencebase.gov/catalog"
USGS_PDF_BASE   = "https://pubs.usgs.gov/periodicals/mcs{year}/mcs{year}-{slug}.pdf"

# Persistent session for connection reuse + polite UA
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "UCSD-MSBA-Capstone-CSI/1.0 (research; contact: capstone team)",
    "Accept": "application/json, text/csv, application/pdf, */*",
})

def http_get(url, **kw):
    """GET with retry + timeout. Returns response or raises."""
    for attempt in range(3):
        try:
            r = SESSION.get(url, timeout=30, **kw)
            if r.status_code == 200:
                return r
            if r.status_code in (429, 503):
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
        except requests.RequestException as e:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after 3 attempts: {url}")

In [ ]:
def find_mcs_parent_item(year: int) -> Optional[str]:
    """Discover the ScienceBase parent item ID for the MCS {year} Data Release.

    Uses the public search API — no hard-coded ID. Returns None if no
    matching release exists yet (e.g. running before USGS publishes that year).

    Search title pattern (stable since 2018):
        'U.S. Geological Survey Mineral Commodity Summaries {year} Data Release'
    """
    url = f"{SCIENCEBASE_API}/items"
    # The 'q' parameter does keyword matching, then we filter results by exact
    # title pattern to avoid false hits (e.g. unrelated commodity publications).
    params = {
        "q": f"Mineral Commodity Summaries {year} Data Release",
        "format": "json",
        "max": 25,
        "fields": "id,title",
    }
    r = http_get(url, params=params)
    items = r.json().get("items", [])

    title_pat = re.compile(
        rf"Mineral Commodity Summaries {year}.*Data Release",
        re.IGNORECASE
    )
    # Prefer the top-level "Data Release" parent, not per-mineral children
    # (which have titles like '... 2025 - COPPER Data Release').
    for it in items:
        title = it.get("title", "")
        if title_pat.search(title) and " - " not in title:
            return it["id"]
    return None


def resolve_mcs_year_and_parent(preferred_year: int) -> tuple[int, str]:
    """Find the most recent MCS edition that's actually published.

    Tries `preferred_year` first, then falls back one year at a time
    (max 3 attempts) if not found. Returns (year_used, parent_id).
    Raises if nothing in the last 3 years is found.
    """
    for offset in range(3):
        year = preferred_year - offset
        pid = find_mcs_parent_item(year)
        if pid:
            if offset > 0:
                print(f"  ⓘ MCS {preferred_year} not yet on ScienceBase; "
                      f"falling back to MCS {year}")
            return year, pid
    raise RuntimeError(
        f"Could not find any MCS Data Release on ScienceBase for years "
        f"{preferred_year-2}..{preferred_year}. Check ScienceBase manually."
    )

In [ ]:
def list_sciencebase_children(parent_id: str) -> pd.DataFrame:
    """Return a DataFrame of all child items under a ScienceBase parent.

    Each MCS commodity (copper, cobalt, etc.) is a separate child item.
    We use the JSON API: /catalog/items?parentId=...&format=json.
    """
    url = f"{SCIENCEBASE_API}/items"
    params = {"parentId": parent_id, "format": "json", "max": 200, "fields": "id,title"}
    r = http_get(url, params=params)
    items = r.json().get("items", [])
    return pd.DataFrame([{"id": i["id"], "title": i["title"]} for i in items])


# --- Resolve parent + cache children once per run ------------------------
# Lazy-initialized: only fires on the first mineral fetch.
SB_PARENT_ID    = None
SB_PARENT_YEAR  = None
SB_CHILDREN_CACHE = None

def _ensure_sb_initialized():
    global SB_PARENT_ID, SB_PARENT_YEAR, SB_CHILDREN_CACHE
    if SB_CHILDREN_CACHE is not None:
        return
    SB_PARENT_YEAR, SB_PARENT_ID = resolve_mcs_year_and_parent(MCS_YEAR_DEFAULT)
    print(f"  ✓ Using ScienceBase parent: {SB_PARENT_ID}  (MCS {SB_PARENT_YEAR})")
    SB_CHILDREN_CACHE = list_sciencebase_children(SB_PARENT_ID)
    print(f"  ✓ Found {len(SB_CHILDREN_CACHE)} child items under this release")


def get_sb_child_for_mineral(mineral: str) -> Optional[str]:
    """Find the ScienceBase child item ID matching a mineral name.

    Title format: 'Mineral Commodity Summaries {year} - COPPER Data Release'
    """
    try:
        _ensure_sb_initialized()
    except Exception as e:
        print(f"⚠ ScienceBase init failed: {e}")
        return None

    # Match by uppercase mineral name in title (year-agnostic)
    pattern = mineral.upper()
    hits = SB_CHILDREN_CACHE[SB_CHILDREN_CACHE["title"].str.contains(
        rf"- {re.escape(pattern)} Data Release", case=False, regex=True)]
    if len(hits) == 0:
        return None
    return hits.iloc[0]["id"]

In [ ]:
def fetch_sciencebase_csvs(item_id: str) -> dict[str, pd.DataFrame]:
    """Download all CSV attachments from a ScienceBase item.

    Returns dict mapping {filename: DataFrame}. Each MCS commodity item
    typically has 2 CSVs:
      - {commodity}_salient_statistics.csv  (US-only — not used by SI3)
      - {commodity}_world_production_reserves.csv  ← what we need
    """
    url = f"{SCIENCEBASE_API}/item/{item_id}"
    r = http_get(url, params={"format": "json"})
    item = r.json()
    csvs = {}
    for f in item.get("files", []):
        name = f.get("name", "")
        if not name.lower().endswith(".csv"):
            continue
        file_url = f.get("url") or f.get("downloadUri")
        if not file_url:
            continue
        try:
            csv_resp = http_get(file_url)
            df = pd.read_csv(pd.io.common.BytesIO(csv_resp.content))
            csvs[name] = df
        except Exception as e:
            print(f"  ⚠ Failed to read {name}: {e}")
    return csvs

In [ ]:
def fetch_usgs_pdf_text(mineral_slug: str, year: int) -> str:
    """Fallback: fetch the MCS PDF and extract text. `year` is required (no default)."""
    url = USGS_PDF_BASE.format(year=year, slug=mineral_slug)
    r = http_get(url)
    pdf_bytes = r.content

    # Lazy-import to avoid hard dependency unless fallback is triggered
    try:
        from pypdf import PdfReader
        from io import BytesIO
        reader = PdfReader(BytesIO(pdf_bytes))
        return "\n".join(p.extract_text() or "" for p in reader.pages)
    except ImportError:
        # pdfplumber is a heavier alternative
        import pdfplumber
        from io import BytesIO
        with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
            return "\n".join(p.extract_text() or "" for p in pdf.pages)

### 2.4 USGS row parsing

USGS MCS world production tables have a recognizable structure:

```
Mine production:                Reserves:
                  2023      2024
United States    1,100    1,100      5,000,000
Chile            5,250    5,300     190,000,000
...
World total   22,300    23,000  1,000,000,000
```

Quirks handled:
- Withheld values appear as `W` → mapped to `NaN` (track separately as `withheld_flag`)
- Footnote superscripts (`²`, `³`, `e`, `r`) are stripped
- Number formatting uses commas; some entries use `<0.5` or `(²)` for negligible values
- Country names sometimes have variants (`United States` vs `U.S.`) — handled by regex aliases

In [ ]:
# --- Number cleaner -----------------------------------------------------
NUMBER_CLEAN_RE = re.compile(r"[,\s]")
WITHHELD_TOKENS = {"W", "w", "—", "-", "NA", "n/a", "(W)"}
NEGLIGIBLE_RE   = re.compile(r"^[\(<]?\s*\d*\.?\d+\s*[\)]?$")

def clean_usgs_number(raw) -> tuple[Optional[float], Optional[str]]:
    """Convert a raw USGS table cell to (value, flag).

    flag is one of: None (clean), 'W' (withheld), 'NEG' (negligible), 'EST' (estimated).
    """
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return (np.nan, None)
    s = str(raw).strip()
    if not s:
        return (np.nan, None)

    # Withheld
    if s in WITHHELD_TOKENS or s.startswith("W"):
        return (np.nan, "W")

    # Strip footnote superscripts and the 'e' (estimated) / 'r' (revised) markers
    flag = None
    if s.endswith("e") or "ᵉ" in s:
        flag = "EST"
        s = s.rstrip("e").replace("ᵉ", "")
    s = re.sub(r"[²³⁴⁵⁶⁷⁸⁹¹⁰ᵃᵇᶜᵈʳ]", "", s)
    s = NUMBER_CLEAN_RE.sub("", s).strip("()")

    if not s:
        return (np.nan, flag)
    try:
        return (float(s), flag)
    except ValueError:
        return (np.nan, flag)


# Quick sanity check
assert clean_usgs_number("1,100") == (1100.0, None)
assert clean_usgs_number("W") == (np.nan, "W")
assert clean_usgs_number("23,000e") == (23000.0, "EST")
assert clean_usgs_number(None)[0] is np.nan or np.isnan(clean_usgs_number(None)[0])
print("✓ Number cleaner sanity checks passed")

In [ ]:
def parse_sciencebase_world_csv(df: pd.DataFrame, mineral: str) -> pd.DataFrame:
    """Normalize a ScienceBase 'world production and reserves' CSV.

    The exact column names vary by mineral, so we detect them by pattern.
    Returns a long-format DataFrame:
        country | year | metric | value | flag | mineral
    """
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Find country column (usually 'Country' or 'country')
    country_col = next((c for c in df.columns if c.lower() in ("country","nation")), df.columns[0])

    # Find production columns (usually 'Mine production 2023', '...2024')
    # and reserves column (usually 'Reserves')
    prod_cols = [c for c in df.columns if re.search(r"production.*20\d{2}|^20\d{2}$", c, re.I)]
    res_cols  = [c for c in df.columns if re.search(r"reserves?", c, re.I)]

    records = []
    for _, row in df.iterrows():
        country = str(row[country_col]).strip()
        if not country or country.lower() in ("world total","total","world"):
            # Capture world total separately for share calculations
            country = "WORLD_TOTAL"

        for c in prod_cols:
            year_match = re.search(r"(20\d{2})", c)
            if not year_match:
                continue
            year = int(year_match.group(1))
            val, flag = clean_usgs_number(row[c])
            records.append({
                "mineral": mineral, "country": country, "year": year,
                "metric": "production", "value": val, "flag": flag,
                "source_col": c,
            })

        for c in res_cols:
            val, flag = clean_usgs_number(row[c])
            # Reserves are reported as "as of end of [latest year]".
            # If we can't infer the year from production columns, fall back
            # to SB_PARENT_YEAR - 1 (MCS{N} reports reserves through end of N-1).
            inferred_years = [int(re.search(r"(20\d{2})", pc).group(1))
                              for pc in prod_cols if re.search(r"(20\d{2})", pc)]
            fallback_year = (SB_PARENT_YEAR - 1) if SB_PARENT_YEAR else TODAY.year - 1
            reserves_year = max(inferred_years, default=fallback_year)
            records.append({
                "mineral": mineral, "country": country, "year": reserves_year,
                "metric": "reserves", "value": val, "flag": flag,
                "source_col": c,
            })

    return pd.DataFrame(records)

In [ ]:
def fetch_one_mineral_usgs(mineral_row) -> pd.DataFrame:
    """Fetch USGS data for one mineral. Tries ScienceBase CSV first, then PDF.

    Uses the year actually resolved by ScienceBase (SB_PARENT_YEAR), which
    may differ from mineral_row['mcs_year'] if a fallback was triggered.

    Returns a long-format DataFrame.
    """
    mineral = mineral_row["mineral"]
    slug    = mineral_row["usgs_slug"]
    requested_year = int(mineral_row["mcs_year"])

    # --- Attempt 1: ScienceBase CSV --------------------------------------
    sb_id = get_sb_child_for_mineral(mineral)
    if sb_id:
        try:
            csvs = fetch_sciencebase_csvs(sb_id)
            # Find the world production CSV
            world_csv = None
            for name, df in csvs.items():
                if re.search(r"world|production.*reserve|reserve", name, re.I):
                    world_csv = df
                    break
            if world_csv is not None:
                parsed = parse_sciencebase_world_csv(world_csv, mineral)
                parsed["source"] = "sciencebase_csv"
                parsed["mcs_year"] = SB_PARENT_YEAR  # actual edition fetched
                return parsed
            else:
                print(f"  ⚠ {mineral}: ScienceBase has no world-production CSV; trying PDF...")
        except Exception as e:
            print(f"  ⚠ {mineral}: ScienceBase fetch failed ({e}); trying PDF...")

    # --- Attempt 2: PDF fallback ----------------------------------------
    # Use SB_PARENT_YEAR if ScienceBase init succeeded (so PDF year matches),
    # else fall back to the mineral_row's requested year.
    pdf_year = SB_PARENT_YEAR if SB_PARENT_YEAR else requested_year
    try:
        text = fetch_usgs_pdf_text(slug, pdf_year)
        # Saves raw text for manual inspection if parsing is incomplete
        (LOG_DIR / f"usgs_{slug}_{pdf_year}.txt").write_text(text)
        # PDF parsing is fragile — emit empty df with marker, do manual cleanup downstream
        print(f"  ⚠ {mineral}: PDF text saved to logs/. Manual extraction needed.")
        return pd.DataFrame(columns=["mineral","country","year","metric","value","flag","source"])
    except Exception as e:
        print(f"  ✗ {mineral}: both ScienceBase and PDF failed: {e}")
        return pd.DataFrame(columns=["mineral","country","year","metric","value","flag","source"])

In [ ]:
# --- Run USGS fetch for all 6 minerals -----------------------------------
usgs_records = []
for _, mrow in tqdm(MINERALS.iterrows(), total=len(MINERALS), desc="USGS minerals"):
    df = fetch_one_mineral_usgs(mrow)
    if not df.empty:
        usgs_records.append(df)
        # Also save per-mineral CSV for traceability
        out = USGS_DIR / f"{mrow['usgs_slug']}_world_raw.csv"
        df.to_csv(out, index=False)

usgs_long = pd.concat(usgs_records, ignore_index=True) if usgs_records else pd.DataFrame()
print(f"\nUSGS rows fetched: {len(usgs_long):,}")
usgs_long.head(15)

---
## 3. Derive USGS metrics

For each (country, mineral), compute:
1. **Production Share** = country production / world total production (latest year)
2. **Reserves Share** = country reserves / world total reserves (latest year)
3. **YoY Production Growth** = (year_t − year_{t-1}) / year_{t-1}

We compute these from the long-format `usgs_long` table.

In [ ]:
def filter_target_countries(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows matching our 6 target countries (by regex).

    USGS spelling varies — use COUNTRIES.usgs_pattern to match.
    """
    if df.empty:
        return df
    out = []
    for _, crow in COUNTRIES.iterrows():
        mask = df["country"].str.match(crow["usgs_pattern"], case=False, na=False)
        sub = df[mask].copy()
        sub["country"] = crow["country"]  # standardize to our internal name
        out.append(sub)
    # Always keep WORLD_TOTAL
    out.append(df[df["country"] == "WORLD_TOTAL"].copy())
    return pd.concat(out, ignore_index=True) if out else df


def compute_usgs_metrics(usgs_long: pd.DataFrame) -> pd.DataFrame:
    """Return wide-format country × mineral metrics derived from USGS.

    Output cols: country, mineral, production_share, reserves_share, yoy_growth
    """
    if usgs_long.empty:
        return pd.DataFrame(columns=["country","mineral","production_share","reserves_share","yoy_growth"])

    df = filter_target_countries(usgs_long)
    latest_year = df["year"].max()

    # --- 3.1 Production share (latest year) -----------------------------
    prod_latest = df[(df["metric"]=="production") & (df["year"]==latest_year)]
    world_prod  = prod_latest[prod_latest["country"]=="WORLD_TOTAL"].set_index("mineral")["value"]
    prod_share  = (prod_latest[prod_latest["country"]!="WORLD_TOTAL"]
                   .merge(world_prod.rename("world_prod"), left_on="mineral", right_index=True))
    prod_share["production_share"] = prod_share["value"] / prod_share["world_prod"]

    # --- 3.2 Reserves share ---------------------------------------------
    res = df[df["metric"]=="reserves"]
    world_res = res[res["country"]=="WORLD_TOTAL"].set_index("mineral")["value"]
    res_share = (res[res["country"]!="WORLD_TOTAL"]
                 .merge(world_res.rename("world_res"), left_on="mineral", right_index=True))
    res_share["reserves_share"] = res_share["value"] / res_share["world_res"]

    # --- 3.3 YoY production growth (latest vs prior year) ----------------
    prior_year = latest_year - 1
    prod_prior = (df[(df["metric"]=="production") & (df["year"]==prior_year)
                     & (df["country"]!="WORLD_TOTAL")]
                  .set_index(["country","mineral"])["value"])
    yoy = prod_share.set_index(["country","mineral"])[["value"]].rename(columns={"value":"prod_curr"})
    yoy["prod_prior"] = prod_prior
    yoy["yoy_growth"] = (yoy["prod_curr"] - yoy["prod_prior"]) / yoy["prod_prior"]

    # --- Combine ---------------------------------------------------------
    out = (prod_share[["country","mineral","production_share"]]
           .merge(res_share[["country","mineral","reserves_share"]],
                  on=["country","mineral"], how="outer")
           .merge(yoy.reset_index()[["country","mineral","yoy_growth"]],
                  on=["country","mineral"], how="outer"))
    return out


usgs_metrics = compute_usgs_metrics(usgs_long)
print(f"USGS metric rows: {len(usgs_metrics)} (expected: up to 36 = 6×6)")
usgs_metrics

---
## 4. UN Comtrade — monthly trade flows

**Goal:** Pull monthly export values for each (country × mineral × stage[raw/processed]).

**API mode auto-switching:**
- If `COMTRADE_KEY` is set → uses `getFinalData()` (250K records per call, no rate limit)
- Otherwise → uses `previewFinalData()` (500 records per call, no key needed)

**Period:** 2020-01 to latest available (Comtrade typically lags 2–3 months).
We request in **annual batches of months** (12 periods at a time) to stay well under
the per-call record cap, even with 5 HS codes per query.

**Reference:**
- `comtradeapicall` PyPI: <https://pypi.org/project/comtradeapicall/>
- API docs: <https://comtradeplus.un.org>

In [ ]:
import comtradeapicall as cta
print(f"comtradeapicall version: {cta.__version__ if hasattr(cta, '__version__') else 'unknown'}")

In [ ]:
# --- Period generator ----------------------------------------------------
def monthly_periods(start_yyyy: int, start_mm: int,
                    end_yyyy: int,   end_mm: int) -> list[str]:
    """Generate ['YYYYMM', ...] inclusive of both endpoints."""
    out = []
    y, m = start_yyyy, start_mm
    while (y, m) <= (end_yyyy, end_mm):
        out.append(f"{y:04d}{m:02d}")
        m += 1
        if m > 12:
            m, y = 1, y + 1
    return out

# Comtrade API caps `period` at 12 values per call when using monthly freq.
# We chunk our full date range into 12-month batches.
def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# --- Auto-detect the latest period to request ----------------------------
# Comtrade typically lags 2-3 months behind the current calendar month.
# Default: request through the month BEFORE last (e.g. on 2026-04-25, ends 2026-02).
# Adjust LAG_MONTHS if you need to reach further back or if you want to
# attempt the very latest months even though some countries may be missing.
LAG_MONTHS = 2
PERIOD_START_YEAR  = 2020   # project history start (CSI methodology decision)
PERIOD_START_MONTH = 1

def _back_month(today: datetime, n_back: int) -> tuple[int, int]:
    """Return (year, month) n_back months before `today`."""
    y, m = today.year, today.month - n_back
    while m <= 0:
        m += 12
        y -= 1
    return y, m

PERIOD_END_YEAR, PERIOD_END_MONTH = _back_month(TODAY, LAG_MONTHS)

ALL_PERIODS = monthly_periods(
    PERIOD_START_YEAR, PERIOD_START_MONTH,
    PERIOD_END_YEAR,   PERIOD_END_MONTH
)
print(f"Total months requested: {len(ALL_PERIODS)} ({ALL_PERIODS[0]} → {ALL_PERIODS[-1]})")
print(f"Number of API calls per (country, HS-bundle): {len(list(chunked(ALL_PERIODS, 12)))}")

In [ ]:
# --- Single-call wrapper that auto-switches preview/premium --------------
def comtrade_call(reporter_m49: str, hs_codes: list[str],
                  periods: list[str], flow: str = "X") -> pd.DataFrame:
    """Fetch one batch of Comtrade data.

    Args:
        reporter_m49: e.g. '842' for USA
        hs_codes: list of 6-digit HS codes, e.g. ['740311','740319']
        periods: up to 12 'YYYYMM' strings
        flow: 'X' = exports (default), 'M' = imports

    Returns DataFrame with columns including:
        period, reporterCode, partnerCode, cmdCode, primaryValue, ...
    """
    cmd_code_str = ",".join(hs_codes)
    period_str   = ",".join(periods)
    common_kwargs = dict(
        typeCode="C",          # Commodities
        freqCode="M",          # Monthly
        clCode="HS",           # Harmonized System
        period=period_str,
        reporterCode=reporter_m49,
        cmdCode=cmd_code_str,
        flowCode=flow,
        partnerCode="0",       # 0 = World (aggregate, no partner breakdown)
        partner2Code=None,
        customsCode=None,
        motCode=None,
        format_output="JSON",
        breakdownMode="classic",
        includeDesc=True,
    )

    if COMTRADE_KEY:
        df = cta.getFinalData(COMTRADE_KEY, maxRecords=250000,
                              aggregateBy=None, countOnly=None, **common_kwargs)
    else:
        # Preview API: capped at 500 records, no key needed.
        # 6 countries × 12 months × 5 HS codes = 360 records max — well under cap.
        df = cta.previewFinalData(maxRecords=500,
                                  aggregateBy=None, countOnly=None, **common_kwargs)

    # API can return None on empty result
    if df is None or len(df) == 0:
        return pd.DataFrame()
    return df

In [ ]:
def fetch_comtrade_for_country_mineral(country_row, mineral: str,
                                       stage: str) -> pd.DataFrame:
    """Fetch all months of one (country, mineral, stage) tuple.

    stage = 'processed' or 'raw'.
    Returns concatenated DataFrame across all monthly batches.
    """
    hs_codes = HS_CODES[mineral][stage]
    batches = list(chunked(ALL_PERIODS, 12))

    results = []
    for batch in batches:
        try:
            df = comtrade_call(country_row["m49"], hs_codes, batch, flow="X")
            if not df.empty:
                df["mineral"] = mineral
                df["stage"]   = stage
                df["country"] = country_row["country"]
                results.append(df)
        except Exception as e:
            print(f"    ⚠ {country_row['country']}/{mineral}/{stage}/{batch[0]}-{batch[-1]}: {e}")

        # Polite delay — avoid hitting Comtrade's rate limiter
        time.sleep(0.6 if not COMTRADE_KEY else 0.2)

    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    return out

In [ ]:
# --- Run Comtrade fetch for all 6×6×2 = 72 combinations ------------------
# This takes ~10–15 min on free tier (rate-limited), ~2 min with premium key.
comtrade_records = []
total_iters = len(COUNTRIES) * len(MINERALS) * 2  # 2 stages

with tqdm(total=total_iters, desc="Comtrade") as pbar:
    for _, crow in COUNTRIES.iterrows():
        for _, mrow in MINERALS.iterrows():
            for stage in ("processed", "raw"):
                pbar.set_postfix_str(f"{crow['country']}/{mrow['mineral']}/{stage}")
                df = fetch_comtrade_for_country_mineral(crow, mrow["mineral"], stage)
                if not df.empty:
                    comtrade_records.append(df)
                    # Save per-tuple CSV for traceability
                    out = COMTRADE_DIR / f"{mrow['usgs_slug']}_{crow['iso3']}_{stage}.csv"
                    df.to_csv(out, index=False)
                pbar.update(1)

if comtrade_records:
    comtrade_long = pd.concat(comtrade_records, ignore_index=True)
else:
    comtrade_long = pd.DataFrame()
print(f"\nComtrade rows fetched: {len(comtrade_long):,}")
(comtrade_long.head() if not comtrade_long.empty else "No data returned")


---
## 5. Derive Comtrade metrics

For each (country, mineral), compute (using the most recent **full year** of monthly data):

1. **Refining Capacity Share** = country processed exports / world processed exports
   (Note: world denominator requires a separate "all reporters" call — see §5.2.)
2. **Value-Add Ratio** = processed_exports / (raw_exports + processed_exports)

In [ ]:
def aggregate_monthly_to_annual(comtrade_long: pd.DataFrame) -> pd.DataFrame:
    """Sum primaryValue (USD) by country × mineral × stage × year."""
    if comtrade_long.empty:
        return pd.DataFrame()

    df = comtrade_long.copy()
    # 'period' format = YYYYMM
    df["year"] = df["period"].astype(str).str[:4].astype(int)
    annual = (df.groupby(["country","mineral","stage","year"], as_index=False)
                .agg(annual_value_usd=("primaryValue","sum")))
    return annual

annual = aggregate_monthly_to_annual(comtrade_long)
(annual.head() if not annual.empty else "No annual data")


In [ ]:
def compute_value_add_ratio(annual: pd.DataFrame) -> pd.DataFrame:
    """Pivot to wide and compute processed/(raw+processed) for the latest full year."""
    if annual.empty:
        return pd.DataFrame(columns=["country","mineral","value_add_ratio","year_used"])

    # Use latest year that has data for at least one stage
    latest = int(annual["year"].max())
    cur = annual[annual["year"]==latest].pivot_table(
        index=["country","mineral"], columns="stage", values="annual_value_usd",
        aggfunc="sum", fill_value=0
    ).reset_index()

    # Ensure both columns exist even if one stage has zero rows
    for col in ("processed","raw"):
        if col not in cur.columns:
            cur[col] = 0.0

    denom = cur["processed"] + cur["raw"]
    cur["value_add_ratio"] = np.where(denom > 0, cur["processed"] / denom, np.nan)
    cur["year_used"] = latest
    return cur[["country","mineral","value_add_ratio","year_used"]]


value_add = compute_value_add_ratio(annual)
value_add

### 5.2 Refining capacity (world denominator)

To compute `country_processed / world_processed`, we need world totals. Two options:

- **Option A (precise):** Fetch all reporters (`reporterCode='all'`) for the same HS codes & period — one call per mineral.
- **Option B (proxy):** Use the sum across our 6 target countries as the denominator. This is **biased low** (other countries also export), so use Option A for production scoring.

We implement **Option A** below. This adds 6 more API calls (1 per mineral) for the latest full year only.

In [ ]:
def fetch_world_processed_exports_latest(year: int) -> pd.DataFrame:
    """Fetch world-aggregate processed exports for each mineral, in `year` only.

    Uses reporterCode='all'. Requires 6 calls (one per mineral),
    each returning ~150-200 country rows × 12 months.
    Premium-key recommended (preview cap = 500 records).
    """
    periods = monthly_periods(year, 1, year, 12)
    rows = []
    for mineral, codes_dict in HS_CODES.items():
        hs_codes = codes_dict["processed"]
        for batch in chunked(periods, 12):
            try:
                kwargs = dict(
                    typeCode="C", freqCode="M", clCode="HS",
                    period=",".join(batch),
                    reporterCode="all",
                    cmdCode=",".join(hs_codes),
                    flowCode="X",
                    partnerCode="0", partner2Code=None,
                    customsCode=None, motCode=None,
                    format_output="JSON", breakdownMode="classic",
                    includeDesc=False,
                )
                if COMTRADE_KEY:
                    df = cta.getFinalData(COMTRADE_KEY, maxRecords=250000,
                                          aggregateBy=None, countOnly=None, **kwargs)
                else:
                    df = cta.previewFinalData(maxRecords=500,
                                              aggregateBy=None, countOnly=None, **kwargs)
                if df is not None and len(df) > 0:
                    df["mineral"] = mineral
                    rows.append(df)
            except Exception as e:
                print(f"  ⚠ world fetch {mineral}/{batch[0]}-{batch[-1]}: {e}")
            time.sleep(0.6 if not COMTRADE_KEY else 0.2)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# Use latest year that appears in our country-level fetch
if not annual.empty:
    latest_full_year = int(annual["year"].max())
    world_df = fetch_world_processed_exports_latest(latest_full_year)
    print(f"World rows: {len(world_df):,}")
else:
    world_df = pd.DataFrame()
    print("Skipping world fetch (no annual data).")

In [ ]:
def compute_refining_share(annual: pd.DataFrame, world_df: pd.DataFrame,
                            year: int) -> pd.DataFrame:
    """Per-mineral country share of world processed exports, USD basis."""
    if annual.empty or world_df.empty:
        return pd.DataFrame(columns=["country","mineral","refining_share","year_used"])

    # Country-level processed total for the year
    cty = (annual[(annual["year"]==year) & (annual["stage"]=="processed")]
           .groupby(["country","mineral"], as_index=False)["annual_value_usd"].sum())

    # World total for the year
    world_df = world_df.copy()
    world_df["year"] = world_df["period"].astype(str).str[:4].astype(int)
    world_tot = (world_df[world_df["year"]==year]
                 .groupby("mineral", as_index=False)["primaryValue"].sum()
                 .rename(columns={"primaryValue":"world_processed_usd"}))

    out = cty.merge(world_tot, on="mineral", how="left")
    out["refining_share"] = out["annual_value_usd"] / out["world_processed_usd"]
    out["year_used"] = year
    return out[["country","mineral","refining_share","year_used"]]


if not annual.empty and not world_df.empty:
    refining = compute_refining_share(annual, world_df, latest_full_year)
else:
    refining = pd.DataFrame(columns=["country","mineral","refining_share","year_used"])
refining

---
## 6. Combine all metrics & save outputs

In [ ]:
# --- Build the master metric panel ---------------------------------------
# Cartesian product of countries × minerals as the canvas, then left-join each metric.
panel = COUNTRIES[["country"]].merge(
    MINERALS[["mineral"]], how="cross"
)

if not usgs_metrics.empty:
    panel = panel.merge(usgs_metrics, on=["country","mineral"], how="left")
else:
    for col in ("production_share","reserves_share","yoy_growth"):
        panel[col] = np.nan

if not refining.empty:
    panel = panel.merge(refining[["country","mineral","refining_share"]],
                        on=["country","mineral"], how="left")
else:
    panel["refining_share"] = np.nan

if not value_add.empty:
    panel = panel.merge(value_add[["country","mineral","value_add_ratio"]],
                        on=["country","mineral"], how="left")
else:
    panel["value_add_ratio"] = np.nan

# --- Save outputs --------------------------------------------------------
panel_path = DATA_DIR / "si3_metric_panel.csv"
panel.to_csv(panel_path, index=False)
print(f"✓ Wrote {panel_path}  ({len(panel)} rows × {len(panel.columns)} cols)")

if not usgs_long.empty or not comtrade_long.empty:
    parts = []
    if not usgs_long.empty:
        usgs_long["source"] = "USGS"
        parts.append(usgs_long.assign(date=usgs_long["year"].astype(str)+"-01-01"))
    if not comtrade_long.empty:
        ct = comtrade_long[["country","mineral","stage","period","primaryValue"]].copy()
        ct["source"] = "Comtrade"
        ct["date"]   = pd.to_datetime(ct["period"].astype(str), format="%Y%m")
        parts.append(ct)
    raw_path = DATA_DIR / "si3_combined_raw.parquet"
    pd.concat(parts, ignore_index=True).to_parquet(raw_path, index=False)
    print(f"✓ Wrote {raw_path}")

panel

---
## 7. Run diagnostics

Quick checks to flag missing data, suspicious values, and data-source escalations.

In [ ]:
def diagnostics(panel: pd.DataFrame) -> None:
    print("=" * 70)
    print(f"WS1 SI3 Run Diagnostics — {RUN_TS}")
    print("=" * 70)

    metric_cols = ["production_share","reserves_share","yoy_growth",
                   "refining_share","value_add_ratio"]

    print("\n— Coverage by metric (non-null cells / total) —")
    for c in metric_cols:
        if c in panel.columns:
            n_ok = panel[c].notna().sum()
            print(f"  {c:25s}: {n_ok:2d} / {len(panel)}  ({n_ok/len(panel)*100:5.1f}%)")

    print("\n— Coverage by country (any metric non-null) —")
    for cty in COUNTRIES["country"]:
        sub = panel[panel["country"]==cty]
        n_ok = sub[metric_cols].notna().any(axis=1).sum()
        print(f"  {cty:12s}: {n_ok}/{len(sub)} mineral rows have ≥1 metric")

    print("\n— Suspicious values —")
    if "production_share" in panel.columns:
        too_high = panel[panel["production_share"] > 1.0]
        if len(too_high):
            print("  ⚠ Production share > 100% (data error?):")
            print(too_high[["country","mineral","production_share"]].to_string(index=False))
    if "value_add_ratio" in panel.columns:
        odd = panel[(panel["value_add_ratio"] < 0) | (panel["value_add_ratio"] > 1)]
        if len(odd):
            print("  ⚠ Value-add ratio outside [0,1]:")
            print(odd[["country","mineral","value_add_ratio"]].to_string(index=False))

    print("\n— Known escalations to verify (from prior project notes) —")
    print("  • Silicon reserves: USGS does not publish; using production share as proxy.")
    print("  • USA withheld values (W) appear in Cobalt, REE production — flagged as NaN.")
    print("  • UAE/Singapore expected zero for most metrics (entrepôt economies).")

diagnostics(panel)

---
## 8. Next steps

This notebook fetches and structures the **raw inputs** for SI3. It does **not** yet:
- Apply min-max normalization to 0–100 scores (per CSI methodology)
- Combine metrics into the SI3 composite using the 0.40/0.30/0.30 weights
- Run ±10pp weight sensitivity tests
- Mineral-weight aggregation (per the project's mineral importance ranking)

Those steps live in the **scoring notebook** (separate file) and should consume `si3_metric_panel.csv` as input.

**Files written:**
- `data/si3_metric_panel.csv` — wide format, ready for scoring
- `data/si3_combined_raw.parquet` — long format with full history (for trend analysis)
- `data/usgs/{slug}_world_raw.csv` — per-mineral USGS raw extracts
- `data/comtrade/{slug}_{iso3}_{stage}.csv` — per-tuple Comtrade extracts
- `data/logs/usgs_{slug}_{year}.txt` — PDF text dumps (only when ScienceBase fallback triggered)